In [1]:
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sympy import true

import part1 # to use code from tutorial
import torch

# Shared Step: Create Datasets with Three Representations

In [2]:
# splitting data into train and test sets and folds for cross-validation

# get the dataframe
df = pd.read_pickle("./sampled_dataframe.pkl")

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["Party_orientation"])

# create cross-validation folds
my_seeds = [0] # we can have more
num_folds = 5

# function to create cross-validation folds
# like in tutorial


In [3]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [4]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] = 1
        return vector

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [5]:
df = df.sample(frac=0.2).reset_index(drop=True)

In [6]:
label_names = sorted(df["Party_orientation"].unique())
column = "lemmas"   # "tokens" or "lemmas"
pairs = make_pairs(df, column, label_names)
vocab = get_vocab([text for text, _ in pairs], min_count=5)

In [7]:
my_cross_validation_sets = []
for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed)
    )

In [8]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in x.split():  # no set() here, so repeated words are counted
            index = self.vocab.get(item)
            if index is not None:
                vector[index] += 1
        return vector

In [9]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [10]:
embeddingDf = pd.read_pickle('./embeddings_sampled.pkl')

In [11]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(embeddingDf)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

In [12]:
my_classifiers["logistic_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.wa

fold 0: accuracy 0.4262
fold 1: accuracy 0.4329
fold 2: accuracy 0.4293
fold 3: accuracy 0.4309
fold 4: accuracy 0.4337


Exp 1.B: Log-Lin with Bag of Words

In [13]:
my_classifiers["logistic_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.wa

fold 0: accuracy 0.4337
fold 1: accuracy 0.4422
fold 2: accuracy 0.4402
fold 3: accuracy 0.4411
fold 4: accuracy 0.4448


Exp 1.C: Log-Lin with Sentence Embeddings

In [14]:
label_names = sorted(embeddingDf["Party_orientation"].unique())
smallEmbeddingDf = embeddingDf.sample(frac=0.1)
column = "sentence"   # "sentence" for embedding pairs
embedding_pairs = make_pairs(smallEmbeddingDf, column, label_names)

# separate folds for the embedding experiments (my_cross_validation_sets holds lemma strings, not sentences)
embedding_cross_validation_sets = []
for seed in my_seeds:
    embedding_cross_validation_sets.append(
        part1.experiment.cross_validation_split(pairs=embedding_pairs, k=num_folds, seed=seed)
    )

In [18]:
my_classifiers["logistic_embeddings"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_embeddings/seed-{seed}"
    my_classifiers["logistic_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=embedding_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="logistic",
            save_to=run_folder,
            loglin_args= {
                'max_iter': 300
            }
        )
    )

accuracies = my_classifiers["logistic_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWar

fold 0: accuracy 0.3906
fold 1: accuracy 0.3904
fold 2: accuracy 0.3820
fold 3: accuracy 0.3951
fold 4: accuracy 0.3829


# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [16]:
my_classifiers["mlp_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.816 seconds
Dataloader done! Time taken: 11.684 seconds
Training...


  0%|          | 0/20 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/part1/smlp.py:71: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/Users/chrisfeltner/Git/nlp/nlp-project/part1/smlp.py:74: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()


Epoch 01 | train loss: 1.7509 | train acc: 0.3308 | val loss: 1.6173 | val acc: 0.3761
Epoch 02 | train loss: 1.5488 | train acc: 0.4127 | val loss: 1.5386 | val acc: 0.4106
Epoch 03 | train loss: 1.4475 | train acc: 0.4609 | val loss: 1.4973 | val acc: 0.4296
Epoch 04 | train loss: 1.3715 | train acc: 0.4932 | val loss: 1.4750 | val acc: 0.4387
Epoch 05 | train loss: 1.3094 | train acc: 0.5181 | val loss: 1.4664 | val acc: 0.4422
Epoch 06 | train loss: 1.2583 | train acc: 0.5390 | val loss: 1.4658 | val acc: 0.4419
Epoch 07 | train loss: 1.2128 | train acc: 0.5550 | val loss: 1.4728 | val acc: 0.4422
Epoch 08 | train loss: 1.1718 | train acc: 0.5707 | val loss: 1.4843 | val acc: 0.4401
Epoch 09 | train loss: 1.1357 | train acc: 0.5824 | val loss: 1.4957 | val acc: 0.4391
Epoch 10 | train loss: 1.1034 | train acc: 0.5944 | val loss: 1.5114 | val acc: 0.4365
Epoch 11 | train loss: 1.0723 | train acc: 0.6057 | val loss: 1.5291 | val acc: 0.4351
Epoch 12 | train loss: 1.0437 | train acc: 

KeyboardInterrupt: 

Exp 2.B: MLP with Bag of Words

In [ ]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

Exp 2.C: MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp_embeddings"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_embeddings/seed-{seed}"
    my_classifiers["mlp_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=embedding_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

# Experiment 3: CNN

Exp 3.A: CNN with Token Indicator

Exp 3.B: CNN with Bag of Words

Exp 3.C: CNN with Sentence Embeddings